# pathintegral_map — exact non-Markovian TEMPO/TNPI map, enhanced-pipeline copy

Exact non-Markovian linear map $\epsilon_{t,0}$ (and its matrix representation $\mathcal L(t)$, eq. 12
of the paper) for a multi-state system bilinearly coupled to independent harmonic baths, computed from
Feynman's path integral with the (discretized) Feynman-Vernon influence functional.

This reimplements the strategy of Seneviratne, Walters & Wang, ACS Omega 2024, 9, 9666 (Sections II &
III), where the exact superoperator is generated with a tensor-network path integral (TNPI, Bose &
Walters, arXiv:2106.12523). Here the augmented path-amplitude tensor is stored as a matrix product
state and propagated TEMPO-style (Strathearn *et al.*, Nat. Commun. 9, 3322 (2018)), with the initial
forward-backward index kept open so that every step directly yields the full $d^2\times d^2$ map matrix
$\mathcal L(t_k)$:
$$\mathrm{vec}(\rho(t_k)) = \mathcal L(t_k)\,\mathrm{vec}(\rho(0))\qquad\text{(column-stacking
convention)}.$$

**Model assumptions** (as in the paper's FMO example):
- each system state $|j\rangle\langle j|$ couples to its own bath (identical spectral densities), i.e.
  coupling operators are the site projectors, which are all diagonal in the site basis;
- factorized initial condition $\rho(0)\otimes e^{-\beta H_{\rm bath}}/Z$.

Units follow the Figures-folder conventions: energies in cm$^{-1}$, $\hbar=1$, time variable
$\tau=2\pi c\,t$ so that phases are $E[{\rm cm}^{-1}] \cdot\tau[{\rm cm}]$.

**This is the ENHANCED-pipeline copy.** The path integral is the only method here that needs a map file
at all: its transfer tensors (see `grid.ipynb`) are learned from the SHORT-time maps
$\mathcal L(\Delta t)..\mathcal L(K\Delta t)$ up to the memory window — NOT the full 0..1000 fs
trajectory — so `compute_and_save` below only computes up to `t_mem_fs`. Everything after that comes
from re-applying the one fixed companion operator, at no further classical cost.

In [1]:
import numpy as np
from scipy.linalg import expm, svd

# no physical constants here: fs_to_cm and kb_cm are passed into
# PathIntegralMap, so main.ipynb is their single source

## Calculating the influence functional $I[s^+, s^-]=e^{{-\frac{1}{\hbar}} \Phi[s^+, s^-]}$

Diese Zelle berechnet die Koeffizenten $a_k$ (als Mtarix für alle $k$) und $a_0$ aus der BCF: $$C_{\text{Drude}}(t) = \underbrace{\lambda\nu \left[ \cot\left(\frac{\beta\nu}{2}\right) - i \right]}_{a_0} e^{-\nu t} + \sum_{k=1}^{\infty} \underbrace{\frac{4\lambda\nu\nu_k}{\beta(\nu_k^2-\nu^2)}}_{a_k} e^{-\nu_k t}$$

In [2]:
def drude_expansion(lam, gamma, beta, n_mats=10000):
    """Exponential decomposition of the bath response function (eq. 10),
    C(tau) = (1/pi) int dw J(w) [coth(beta w/2) cos(w tau) - i sin(w tau)]
           = sum_m a_m exp(-nu_m tau)   for tau >= 0  (Matsubara series).

    Zerlegt die komplexe Bad-Korrelationsfunktion C(tau) in eine Summe von
    abfallenden Exponentialfunktionen. Das ist mathematisch noetig, um die
    Integrale fuer das Influence Functional (eta) analytisch loesen zu koennen.
    """
    n = np.arange(1, n_mats + 1)
    nu = 2 * np.pi * n / beta
    a = (4 * lam * gamma / beta) * nu / (nu ** 2 - gamma ** 2)
    a0 = lam * gamma * (1.0 / np.tan(beta * gamma / 2) - 1j)
    return np.concatenate([[a0], a]), np.concatenate([[gamma], nu])

Diese Zelle berechnet die Terme $\eta_0$ und $\eta_{\Delta_k}$ die für das $\Phi$ im Influence functional $I[s^+, s^-]=e^{{-\frac{1}{\hbar}} \Phi[s^+, s^-]}$ in der nächsten Zelle gebraucht werden: $$\eta_0 = \sum_m \frac{a_m}{\nu_m^2}\Big(\nu_m\Delta t + e^{-\nu_m\Delta t} - 1\Big),\qquad \eta_{\Delta k} = \sum_m \frac{a_m}{\nu_m^2}\,e^{-\nu_m(\Delta k - 1)\Delta t}\Big(1 - e^{-\nu_m\Delta t}\Big)^2\ \ (\Delta k \ge 1).$$


In [ ]:
def eta_coefficients(a, nu, dtau, kmax):
    """
    Parameters:
        a: Die Amplituden (Gewichte) $a_m$ der exponentiellen Zerlegung aus der Matsubara-Expansion.
        nu: Die Dämpfungsraten (Frequenzen) nu_m der Bad-Moden.
        dtau: Die diskrete Zeitscheiben-Länge Delta_tau (Grid-Größe) in den passenden physikalischen Einheiten (hier in Zentimetern laut Konvention)
        kmax: Die maximale Gedächtnislänge K (Fenstergröße)

    Discretized influence-functional coefficients (QUAPI/TEMPO form):
       eta[0] = int_0^dt ds int_0^s  ds' C(s - s')
       eta[k] = int_0^dt ds int_0^dt ds' C(k*dt + s - s'),  k >= 1
    evaluated analytically for C(tau) = sum_m a_m exp(-nu_m tau).

    Berechnet die eta-Werte fuer die nicht-lokalen Zeit-Tiles.
    eta[0] ist die rein lokale Bad-Wirkung am selben Zeitschritt.
    eta[k] ist das "Gedaechtnis" ueber k Zeitschritte hinweg (Kopplung
    zwischen alpha_m und alpha_{m-k}).
    """
    eta = np.empty(kmax + 1, dtype=complex)
    x = nu * dtau
    eta[0] = np.sum(a * (x + np.expm1(-x)) / nu ** 2)   # np.expm1(x) berechnet e^x - 1 mit einer Taylorreihe um keine Nachkommastellen zu verlieren für sehr kleine x
    base = a * np.expm1(-x) ** 2 / nu ** 2              # a_m (1 - e^{-nu dt})^2 / nu^2
    for k in range(1, kmax + 1):
        eta[k] = np.sum(base * np.exp(-nu * (k - 1) * dtau))
    return eta

Diese zelle berechnet die Phase $\Phi$ des Influence functionals mit den $\eta_0$ und  $\eta_{\Delta k}$ von zuvor:
$$\Phi_{k}[s^+, s^-]=-\big [ \eta_{\Delta k}\,\delta_{s^+_k,s^+_{k'}} - \eta^*_{\Delta k}\,\delta_{s^+_k,s^-_{k'}} - \eta_{\Delta k}\,\delta_{s^-_k,s^+_{k'}} + \eta^*_{\Delta k}\,\delta_{s^-_k,s^-_{k'}} \big ]$$

In [4]:
def influence_matrices(eta, d):
    """
    Wandelt die abstrakten eta-Skalare in reale Einfluss-Tensoren I_D um.
    Da wir im Liouville-Raum sind, kombiniert alpha den Vorwaerts- und
    Rueckwaertspfad (s_plus, s_minus). Die Funktion gibt eine Liste von
    Matrizen zurueck, wobei mats[k] der Tensor I_k(alpha_k, alpha_{k-k'}) ist.
    """
    D = d * d   # Gesamtdimension des Liouville-Raums (z.B. 4*4 = 16)

    # sp (s_plus) extrahiert den Vorwärts-Zustand (Ket) für jeden der D Super-Indizes.
    # Beispiel für d=4: [0, 1, 2, 3, 0, 1, 2, 3, ...] -> Modulo-Operation
    sp = np.arange(D) % d

    # sm (s_minus) extrahiert den Rückwärts-Zustand (Bra) für jeden der D Super-Indizes.
    # Beispiel für d=4: [0, 0, 0, 0, 1, 1, 1, 1, ...] -> Ganzzahl-Division
    sm = np.arange(D) // d

    # --- HIER STECKEN DIE DELTA-FUNKTIONEN (Kronecker-Deltas) ---
    # Durch das Broadcasting [:, None] == [None, :] entsteht jeweils eine D x D Matrix.
    # Ein Eintrag (i, j) ist 1.0 wenn die Bedingung wahr ist, sonst 0.0.
    dpp = (sp[:, None] == sp[None, :]).astype(float)     # dpp = delta(s+_später, s+_früher) -> Kopplung Vorwärts mit Vorwärts
    dpm = (sp[:, None] == sm[None, :]).astype(float)     # dpm = delta(s+_später, s-_früher) -> Kopplung Vorwärts mit Rückwärts
    dmp = (sm[:, None] == sp[None, :]).astype(float)     # dmp = delta(s-_später, s+_früher) -> Kopplung Rückwärts mit Vorwärts
    dmm = (sm[:, None] == sm[None, :]).astype(float)     # dmm = delta(s-_später, s-_früher) -> Kopplung Rückwärts mit Rückwärts

    mats = []
    # Schleife über alle eta-Koeffizienten (für jeden zeitlichen Abstand k)
    for e in eta:
        # phi berechnet den Exponenten der Feynman-Vernon-Formel. Hier werden die Delta-Matrizen mit den komplexen eta-Werten skaliert.
        phi = -(e * dpp - np.conj(e) * dpm - e * dmp + np.conj(e) * dmm)
        mats.append(np.exp(phi)) # I = np.exp(Phi). np.exp wird elementweise auf die D x D Matrix angewendet. Form: I[alpha_später, alpha_früher]

    return mats

## MPS helpers

### SVD und QR-Algorithmus:
Das Produkt $\text{Gesamt}$ ist viel zu groß, um es auf einmal auszurechnen (die inneren Indizes $y_k$ würden exponentiell wachsen). Deshalb wollen wir durch eine truncated SVD die Matrixmultiplikation über kleinere Dimensionen machen. Die Dimension des Matrixproduktes verändert sich nicht. 

- Die SVD $X = U \Sigma V^\dagger$ findet die Richtung der größten Varianz. Die $\vec{u_i}$ aus $U = (\vec{u_1},...,\vec{u_k})$ sind genau diese Richtungen. Da $W$ für eine ganze SVD zu groß ist machen wir eine rSVD. Dabei multiplizieren wir mit einer random Gaussmatrix $G$ (deren Spaltenzahl die Zahl der Singulärwerte bestimmt mit denen wir $W$ annähern wollen) und erhlaten die Matrix $Y = W W^\dagger W G = (U \Sigma V^\dagger)(V \Sigma U^\dagger) (U \Sigma V^\dagger) G = U \Sigma^3 V^\dagger$. Die größten Singulärwerte dominieren und lassen die Spaltenektoren $y_i$ in $Y$ in die entsprechende Richtung der größten Varianz zeigen: $\vec{y}_{\text{power}} = \sum_{i=1}^n (\sigma_i^3 \cdot g'_i) \cdot \vec{u}_i$.

- Die $y_i$ zeigen alle in eine ähnliche Richtung (die der größten Varianz). Die QR-Zerlegung $Q, R = \text{qr}(Y)$ nuzt Gram-Schmidt ($\vec{q}_1 = \frac{\vec{y}_1}{\lVert\vec{y}_1\rVert}, \vec{v}_2 = \vec{y}_2 - (\vec{q}_1^\dagger \vec{y}_2)\vec{q}_1 \quad$ $\rightarrow \quad \vec{q}_2 = \frac{\vec{v}_2}{\lVert\vec{v}_2\rVert}$ und macht aus den $y_i$ eine orthonormal Basis $Q = (\vec{q_1},...,\vec{q_k}), \quad  Q^\dagger Q = I$)$. Die obere Dreiecksmatrix R enthält die Gewichte für den Übergang von $Y$ nach $Q$: $\begin{pmatrix} \vert & \vert & \vert \\ \vec{y}_1 & \vec{y}_2 & \vec{y}_3 \\ \vert & \vert & \vert \end{pmatrix} = \begin{pmatrix} \vert & \vert & \vert \\ \vec{q}_1 & \vec{q}_2 & \vec{q}_3 \\ \vert & \vert & \vert \end{pmatrix} \cdot \begin{pmatrix} r_{11} & r_{12} & r_{13} \\ 0 & r_{22} & r_{23} \\ 0 & 0 & r_{33} \end{pmatrix}$

- Da $Q$ den Bildraum von $W$ (fast) perfekt einfängt, gilt die Näherung:$W \approx Q Q^\dagger W$. $B = Q^\dagger W$ projeziert die große Matrix $W$ mit der neu gefundenen Basis $Q$ auf ein kleines $B$. Da $B$ so klein ist, können wir eine exakte SVD machen: $B = U_b \Sigma_b V_b^\dagger$. Daraus folgt: $W \approx Q \cdot B = Q \cdot (U_b \Sigma_b V_b^\dagger) = (Q U_b) \Sigma_b V_b^\dagger = \sum_{i=1}^k \tilde{\sigma}_i \cdot \tilde{\vec{u}}_i \cdot \tilde{\vec{v}}_i^\dagger$

- Wir können die große Matrix $W$ also durch die $k$ wichtigsten Singulärwerte annähern. Das ist uns aber nicht genung. Wir wollen die größten $\chi \leq k$ singulärwerte finden um $W_\chi \approx \sum_{i=1}^\chi \tilde{\sigma}_i \cdot \tilde{\vec{u}}_i \cdot \tilde{\vec{v}}_i^\dagger$ noch gröber zu approximieren. Dazu nutzen wir $\frac{\lVert W-W_\chi\rVert_F^2}{\lVert W \rVert_F^2} = \frac{\;\left\lVert \sum_{k=1}^{r} \sigma_k u_k v_k^\dagger - \sum_{k=1}^{\chi} \sigma_k u_k v_k^\dagger \right\rVert_F^2\;}{\left\lVert \sum_{k=1}^{r} \sigma_k u_k v_k^\dagger \right\rVert_F^2} = \frac{\;\left\lVert \sum_{k>\chi}^{r} \sigma_k u_k v_k^\dagger \right\rVert_F^2\;}{\left\lVert \sum_{k=1}^{r} \sigma_k u_k v_k^\dagger \right\rVert_F^2} = \frac{\sum_{k>\chi} \sigma_k^2}{\sum_{k=1}^r \sigma_k^2} \ \le\ \varepsilon^2 $ was zu einem Kriterium für $\chi$ umgeformt werden kann: $\chi=\min\Big\{\,\chi' \in\{1,\dots,\chi_{max}\}\ :\ \sum_{k>\chi'}\sigma_k^2\ \le\ \varepsilon^2\sum_{k}\sigma_k^2\,\Big\}$. Die Zahl $\chi$ gibt an, wie viele der kleinsten Singulärwerte du wegwerfen darfst, sodass die Ungleichung für ein gegebenes $\epsilon$ gerade noch stimmt.

- Wir können die Matrix $W$ durch diese SVD in das Produkt $\text{SVD}\Big[W^{(1)}_{\alpha_0, \alpha_1, y_1}\Big] \approx \sum_{c_1=1}^{\chi} \underbrace{U^{(1)}_{(\alpha_0, \alpha_1), c_1}}_{G^{(1)}_{\alpha_0, \alpha_1, c_1}} \underbrace{\Sigma^{(1)}_{c_1} (V^\dagger)^{(1)}_{c_1, y_1}}_{C^{(1)}_{c_1, y_1}} \equiv \sum_{c_1=1}^{\chi} G^{(1)}_{\alpha_0, \alpha_1, c_1} C^{(1)}_{c_1, y_1}$ umschreiben. Das erlaubt uns durch geschickets Umordnen der Matrizen eine günstigere der Multiplikation der $W$ im $\text{Gesamt}$-Term: 
$\sum_{ {\color{#FF5555}{y_1}}, {\color{#66CCFF}{y_2}} \dots} \underbrace{\underset{(256 \times {\color{#FF5555}{16}})}{W^{(1)}_{\alpha_0, \alpha_1, {\color{#FF5555}{y_1}}}}}_{\text{SVD}} \cdot \underset{({\color{#FF5555}{16}} \times 16 \times {\color{#66CCFF}{256}})}{W^{(2)}_{{\color{#FF5555}{y_1}}, \alpha_2, {\color{#66CCFF}{y_2}}}} \cdot \underset{({\color{#66CCFF}{256}} \times 16 \times {\color{#55FF55}{4096}})}{W^{(3)}_{{\color{#66CCFF}{y_2}}, \alpha_3, {\color{#55FF55}{y_3}}}} \dots \approx \sum_{ {\color{#FF5555}{y_1}}, {\color{#66CCFF}{y_2}} \dots} \sum_{ {\color{#D388FF}{c_1}}=1}^{{\color{#D388FF}{\chi_1}}} \Big( \underset{(256 \times {\color{#D388FF}{\chi_1}})}{G^{(1)}_{(\alpha_0, \alpha_1), {\color{#D388FF}{c_1}}}} \cdot \underset{({\color{#D388FF}{\chi_1}} \times {\color{#FF5555}{16}})}{C^{(1)}_{{\color{#D388FF}{c_1}}, {\color{#FF5555}{y_1}}}} \Big) \cdot \underset{({\color{#FF5555}{16}} \times 16 \times {\color{#66CCFF}{256}})}{W^{(2)}_{{\color{#FF5555}{y_1}}, \alpha_2, {\color{#66CCFF}{y_2}}}} \cdot \underset{({\color{#66CCFF}{256}} \times 16 \times {\color{#55FF55}{4096}})}{W^{(3)}_{{\color{#66CCFF}{y_2}}, \alpha_3, {\color{#55FF55}{y_3}}}} \dots \qquad \qquad \quad$

In [ ]:
_rng = np.random.default_rng(12345)

def _truncate_rank(sigma, eps, chi_max, total2):
    """Number of singular values to keep so that the discarded Frobenius
    weight (incl. any uncomputed tail, total2 - sum sigma^2) stays <= eps^2.

    Implementiert das Eckart-Young Kriterium zum Finden der effektiven
    Bond-Dimension chi. Wirft alle kleinsten Singulaerwerte weg, solange
    die Summe ihrer Quadrate (der Fehler) unter dem Schwellenwert eps liegt.
    Der unberechnete 'tail' (aus der rSVD) wird fairerweise mitgezaehlt.
        sigma: Singulaerwerte(absteigend sortiert)
        eps: relative Fehlergrenze (z.B. 1e-6)
        chi_max: maximale erlaubte Bond-Dimension (z.B. 1000)
        total2: Frobenius-Norm des Originaltensors (vor der SVD): ||Psi||_F^2 = sum_k |sigma_k|^2
    """

    sigma2 = sigma ** 2                         # Berechnet die Quadrate aller (berechneten) Singulärwerte sigma_k^2

    # Falls eine randomisierte SVD zum berechnen der Singulärwerte s verwendet wurde, wurden manche sehr kleinen Singulärwerte gar nicht erst 
    # berechnet. Dieser unberechnete Rest ("Tail") wird hier ermittelt.
    tail = max(total2 - sigma2.sum(), 0.0)

    # Bildet die Summe beginnend mit dem quadrat des kleinsten (sigma2[::-1]: Dreht das Array um) Singulärwerts (inklusive des Rests). 
    # csum ist dann ein array mit einträgen: tail + [sigma2[-1], sigma2[-1] + sigma2[-2], ... , sigma2[-1] + ... + sigma2[0]]
    # Der Eintrag csum[m] gibt also an, wie groß der quadratische Fehler wäre, wenn du die m kleinsten Singulärwerte wegwirfst.
    csum = tail + np.cumsum(sigma2[::-1])

    # np.searchsorted(..., side='right') sucht die Position in der ansteigenden Fehler-Summe csum. Es sagt dir: "Bis zu welchem Index ist 
    # die Summe der Quadrate noch <= epsilon^2 * total2?"
    # Das Ergebnis ndiscard ist die maximale Anzahl an Elementen, die wir von hinten (den kleinsten Werten) wegwerfen dürfen, ohne das 
    # Fehlerbudget zu verletzen.
    ndiscard = np.searchsorted(csum, eps ** 2 * total2, side='right')

    # Wenn wir wissen, wie viele wir maximal wegwerfen dürfen (ndiscard), dann ist die Anzahl der Werte, die wir mindestens behalten müssen, 
    # um die Bedingung zu erfüllen: chi = Anzahl aller Werte - ndiscard. max sorgt dafür dass immer mindestens der größte Singulärwert behalten wird.
    keep = max(1, sigma.size - ndiscard)

    # sorgt dafür, dass keine Singulärwerte behalten werden die kleiner als 1e-14 sind. np.sum(sigma > 1e-14 * sigma[0]) zählt wie viel
    # Singulärwerte größer als 1e-14 * sigma[0] sind. Das "or 1" sorgt dafür, dass mindestens 1 Singulärwert behalten wird, falls alle zu klein sind.
    keep = min(keep, int(np.sum(sigma > 1e-14 * sigma[0])) or 1)

    # Die mathematische Formel verlangt ganz vorne ein Minimum mit chi_max. Es gilt aber quasi immer chi < chi_max
    return min(keep, chi_max)

### Singular value decomposition (SVD)

In [ ]:
def _full_svd(M):
    """Klassische, exakte Singulaerwertzerlegung als Fallback."""
    try:
        return svd(M, full_matrices=False, lapack_driver='gesdd') # gsdd ist extrem schnell aber instabil 
    except np.linalg.LinAlgError:                                 # Falls es error gibt verwenden wir langsamen stabilen Algorithmus
        print("Warning: SVD failed with gesdd, falling back to gesvd (slower but stable).")
        return svd(M, full_matrices=False, lapack_driver='gesvd') # gesvd ist langsam, dafür aber stabil


def _svd_split(M, eps, chi_max, rank_hint=None):
    """Truncated SVD of M with relative Frobenius-norm cutoff eps.
    Uses a randomized range-finder (one power iteration) when M is large
    and a modest rank is expected; falls back to exact LAPACK SVD.

    Das ist die Halko-Martinsson-Tropp rSVD Implementierung.
    """
    m, n = M.shape
    total2 = np.linalg.norm(M) ** 2  # Exakte Gesamtmasse fuer den Tail-Term: ||M||_F^2 = sum_ij ||M_ij||^2 = sum_k ||sigma_k||^2, für alle k

    # Prüft, ob die Matrix gross genug ist, dass sich die rSVD überhaupt lohnt (ist für kleine Matrizen langsmer als exakte SVD)
    # rSVD nur wenn: 
    # - Fehlerbudget epsilon > 0 existiert (sonst bräuchten wir zwingend eine exakte SVD), 
    # - Die kürzeste Seite der Matrix größer als 512 ist.
    # - Wir eine grobe Schätzung (rank_hint) haben, wie viele Singulärwerte wir behalten wollen (z.B. aus vorherigen Zeitschritt der Simulation)
    if eps > 0 and min(m, n) > 512 and rank_hint is not None:
        Mh = M.conj().T

        # Wenn wir schätzen, dass wir rank_hint Werte behalten, berechnen wir absichtlich etwas mehr 
        # (Faktor 1.3 plus einen Puffer von 64). Dieses Oversampling ist entscheidend in der rSVD, damit die Schätzung für die größten 
        # Werte wirklich hochpräzise wird. Die min(...) Bedingungen verhindern lediglich, dass k größer wird als die Matrix selbst.
        k = min(int(1.3 * rank_hint) + 64, min(m, n), chi_max + 32) # Oversampling

        # Diese Schleife erlaubt es dem Algorithmus, seinen Versuch mit einem größeren k zu wiederholen, falls die erste Schätzung zu klein 
        # war. Sie bricht ab, wenn k 40% der Matrixgröße übersteigt (dann wird rSVD ineffizient und eine exakte SVD ist ohnehin besser).
        while k < 0.4 * min(m, n):
            # 1. Skizze (Random Gauss Matrix G)
            # Erstellt eine Zufallsmatrix G aus Gaußschen Zahlen (Real- und Imaginärteil). Diese Matrix fungiert als eine Art "Netz", das 
            # zufällig in den Spaltenraum von M geworfen wird. Die Spaltenzahl k von G gibt die Zahl der Basisvektoren in der rSVD an.
            G = (_rng.standard_normal((n, k)) + 1j * _rng.standard_normal((n, k)))

            # 2. Power Iteration (Verstaerkt dominante Sigma-Werte)
            # Die Multiplikation mit M M^\dagger dämpft extrem kleine Singulärwerte ab und hebt die großen (wichtigen) hervor. Dadurch 
            # "drehen" sich die zufälligen Spalten von G viel stärker in die Richtungen, die die wesentliche Information von M tragen. 
            # Das reduziert Fehler immens.
            Y = M @ (Mh @ (M @ G))            # one power iteration

            # 3. Range-Finder (Findet die Orthonormalbasis Q)
            # Die QR-Zerlegung orthonormalisiert die Spalten von Y. Die Matrix Q bildet nun eine sehr gute (und winzige!) Basis für den 
            # wichtigsten Unterraum von M.
            Q, _ = np.linalg.qr(Y)

            # 4. Projektion auf kleinen Unterraum B und exakte SVD von B
            # Die riesige Matrix M wird in den durch Q gefundenen kleinen Raum projiziert (B = Q^\dagger M). B ist nun eine sehr kleine 
            # Matrix der Größe k × n.
            B = Q.conj().T @ M

            # Da B so klein ist, können wir hier blitzschnell eine exakte, klassische SVD durchführen.
            Ub, s, Vh = _full_svd(B)

            # Wir rufen unsere bekannte Funktion auf, um zu schauen, wie viele dieser approximierten Singulärwerte s wir behalten 
            # müssen (keep), um unseren Fehler epsilon bezogen auf total2 einzuhalten.
            keep = _truncate_rank(s, eps, chi_max, total2)

            # Adaptivitaet: War unsere Schaetzung 'k' gut genug?
            # Wenn wir k Singulärwerte berechnet haben, aber nur keep Stück behalten (wobei wir mindestens die 8 kleinsten nicht behalten), 
            # wissen wir: Wir haben den Rand unseres Spektrums (den tail) sicher erreicht. Die Approximation war perfekt.
            if keep < k - 8 or k >= chi_max:   # tail captured -> accept
                # Wir schneiden die Arrays auf :keep ab. Um die linke unitäre Matrix U wieder in den originalen, großen Raum von $M$ 
                # anzuheben, multiplizieren wir das kleine $U_b$ wieder mit unserer Basis Q (Q @ Ub).
                return (Q @ Ub)[:, :keep], s[:keep], Vh[:keep]      # ist eine approximation für M

            # Wenn nicht, verdopple die Skizzen-Groesse und versuche es erneut
            k = min(2 * k, min(m, n))          # sketch too small: retry

    # Fallback: Matrix ist zu klein fuer rSVD, mache exakte SVD
    U, s, Vh = _full_svd(M)
    keep = _truncate_rank(s, eps, chi_max, total2) if s.size else 1
    return U[:, :keep], s[:keep], Vh[:keep]

## TEMPO propagation of the map

$$\begin{aligned}  \text{Gesamt} &= \sum_{ {\color{#FF5555}{y_1}}, {\color{#66CCFF}{y_2}} \dots} \underbrace{\underset{(256 \times {\color{#FF5555}{16}})}{W^{(1)}_{\alpha_0, \alpha_1, {\color{#FF5555}{y_1}}}}}_{\text{SVD}} \cdot \underset{({\color{#FF5555}{16}} \times 16 \times {\color{#66CCFF}{256}})}{W^{(2)}_{{\color{#FF5555}{y_1}}, \alpha_2, {\color{#66CCFF}{y_2}}}} \cdot \underset{({\color{#66CCFF}{256}} \times 16 \times {\color{#55FF55}{4096}})}{W^{(3)}_{{\color{#66CCFF}{y_2}}, \alpha_3, {\color{#55FF55}{y_3}}}} \dots \\  &\approx \sum_{ {\color{#FF5555}{y_1}}, {\color{#66CCFF}{y_2}} \dots} \sum_{ {\color{#D388FF}{c_1}}=1}^{{\color{#D388FF}{\chi_1}}} \Big( \underset{(256 \times {\color{#D388FF}{\chi_1}})}{G^{(1)}_{(\alpha_0, \alpha_1), {\color{#D388FF}{c_1}}}} \cdot \underset{({\color{#D388FF}{\chi_1}} \times {\color{#FF5555}{16}})}{C^{(1)}_{{\color{#D388FF}{c_1}}, {\color{#FF5555}{y_1}}}} \Big) \cdot \underset{({\color{#FF5555}{16}} \times 16 \times {\color{#66CCFF}{256}})}{W^{(2)}_{{\color{#FF5555}{y_1}}, \alpha_2, {\color{#66CCFF}{y_2}}}} \cdot \underset{({\color{#66CCFF}{256}} \times 16 \times {\color{#55FF55}{4096}})}{W^{(3)}_{{\color{#66CCFF}{y_2}}, \alpha_3, {\color{#55FF55}{y_3}}}} \dots \\  &= \sum_{ {\color{#66CCFF}{y_2}}, {\color{#55FF55}{y_3}} \dots} \sum_{ {\color{#D388FF}{c_1}}=1}^{{\color{#D388FF}{\chi_1}}} \underset{(256 \times {\color{#D388FF}{\chi_1}})}{G^{(1)}_{(\alpha_0, \alpha_1), {\color{#D388FF}{c_1}}}} \cdot \underbrace{\Big( \sum_{{\color{#FF5555}{y_1}}} \underset{({\color{#D388FF}{\chi_1}} \times {\color{#FF5555}{16}})}{C^{(1)}_{{\color{#D388FF}{c_1}}, {\color{#FF5555}{y_1}}}} \cdot \underset{({\color{#FF5555}{16}} \times 16 \times {\color{#66CCFF}{256}})}{W^{(2)}_{{\color{#FF5555}{y_1}}, \alpha_2, {\color{#66CCFF}{y_2}}}} \Big)}_{\substack{\equiv M^{(2)}_{{\color{#D388FF}{c_1}}, \alpha_2, {\color{#66CCFF}{y_2}}} \rightarrow \text{SVD} \\ \mathbf{( ({\color{#D388FF}{\chi_1}} \cdot 16) \times {\color{#66CCFF}{256}} )}}} \cdot \underset{({\color{#66CCFF}{256}} \times 16 \times {\color{#55FF55}{4096}})}{W^{(3)}_{{\color{#66CCFF}{y_2}}, \alpha_3, {\color{#55FF55}{y_3}}}} \dots \\  &\approx \sum_{ {\color{#66CCFF}{y_2}}, {\color{#55FF55}{y_3}} \dots} \sum_{ {\color{#D388FF}{c_1}}=1}^{{\color{#D388FF}{\chi_1}}} \sum_{ {\color{#00FFFF}{c_2}}=1}^{{\color{#00FFFF}{\chi_2}}} \underset{(256 \times {\color{#D388FF}{\chi_1}})}{G^{(1)}_{(\alpha_0, \alpha_1), {\color{#D388FF}{c_1}}}} \cdot \Big( \underset{(({\color{#D388FF}{\chi_1}} \cdot 16) \times {\color{#00FFFF}{\chi_2}})}{G^{(2)}_{({\color{#D388FF}{c_1}}, \alpha_2), {\color{#00FFFF}{c_2}}}} \cdot \underset{({\color{#00FFFF}{\chi_2}} \times {\color{#66CCFF}{256}})}{C^{(2)}_{{\color{#00FFFF}{c_2}}, {\color{#66CCFF}{y_2}}}} \Big) \cdot \underset{({\color{#66CCFF}{256}} \times 16 \times {\color{#55FF55}{4096}})}{W^{(3)}_{{\color{#66CCFF}{y_2}}, \alpha_3, {\color{#55FF55}{y_3}}}} \dots \\  &= \sum_{ {\color{#55FF55}{y_3}} \dots} \sum_{ {\color{#D388FF}{c_1}}=1}^{{\color{#D388FF}{\chi_1}}} \sum_{ {\color{#00FFFF}{c_2}}=1}^{{\color{#00FFFF}{\chi_2}}} \underset{(256 \times {\color{#D388FF}{\chi_1}})}{G^{(1)}_{(\alpha_0, \alpha_1), {\color{#D388FF}{c_1}}}} \cdot \underset{(({\color{#D388FF}{\chi_1}} \cdot 16) \times {\color{#00FFFF}{\chi_2}})}{G^{(2)}_{({\color{#D388FF}{c_1}}, \alpha_2), {\color{#00FFFF}{c_2}}}} \cdot \underbrace{\Big( \sum_{{\color{#66CCFF}{y_2}}} \underset{({\color{#00FFFF}{\chi_2}} \times {\color{#66CCFF}{256}})}{C^{(2)}_{{\color{#00FFFF}{c_2}}, {\color{#66CCFF}{y_2}}}} \cdot \underset{({\color{#66CCFF}{256}} \times 16 \times {\color{#55FF55}{4096}})}{W^{(3)}_{{\color{#66CCFF}{y_2}}, \alpha_3, {\color{#55FF55}{y_3}}}} \Big)}_{\substack{\equiv M^{(3)}_{{\color{#00FFFF}{c_2}}, \alpha_3, {\color{#55FF55}{y_3}}} \rightarrow \text{SVD} \\ \mathbf{( ({\color{#00FFFF}{\chi_2}} \cdot 16) \times {\color{#55FF55}{4096}} )}}} \dots \\  &\quad \vdots \\  &= \sum_{ {\color{#FF66FF}{y_N}} } \sum_{ {\color{#D388FF}{c_1}} \dots {\color{#FF9999}{c_{N-1}}} }^{ {\color{#D388FF}{\chi_1}} \dots {\color{#FF9999}{\chi_{N-1}}} } \underset{(256 \times {\color{#D388FF}{\chi_1}})}{G^{(1)}_{\dots}} \cdots \underset{(({\color{#FFFF55}{\chi_{N-2}}} \cdot 16) \times {\color{#FF9999}{\chi_{N-1}}})}{G^{(N-1)}_{\dots}} \cdot \underbrace{\Big( \sum_{{\color{#FFB347}{y_{N-1}}}} \underset{({\color{#FF9999}{\chi_{N-1}}} \times {\color{#FFB347}{\text{riesig}}})}{C^{(N-1)}_{{\color{#FF9999}{c_{N-1}}}, {\color{#FFB347}{y_{N-1}}}}} \cdot \underset{({\color{#FFB347}{\text{riesig}}} \times 16 \times \mathbf{{\color{#FF66FF}{1}}})}{W^{(N)}_{{\color{#FFB347}{y_{N-1}}}, \alpha_N, {\color{#FF66FF}{y_N}}}} \Big)}_{\substack{\equiv M^{(N)}_{{\color{#FF9999}{c_{N-1}}}, \alpha_N, {\color{#FF66FF}{y_N}}} \rightarrow \text{SVD} \\ \mathbf{( ({\color{#FF9999}{\chi_{N-1}}} \cdot 16) \times {\color{#FF66FF}{1}} )}}} \\  &\approx \sum_{ {\color{#D388FF}{c_1}} \dots {\color{#BBFF66}{c_N}} }^{ {\color{#D388FF}{\chi_1}} \dots {\color{#BBFF66}{\chi_N}} } \underbrace{G^{(1)}_{\alpha_0, \alpha_1, {\color{#D388FF}{c_1}}}}_{(16 \times 16 \times {\color{#D388FF}{\chi_1}})} \cdot \underbrace{G^{(2)}_{{\color{#D388FF}{c_1}}, \alpha_2, {\color{#00FFFF}{c_2}}}}_{({\color{#D388FF}{\chi_1}} \times 16 \times {\color{#00FFFF}{\chi_2}})} \cdot \underbrace{G^{(3)}_{{\color{#00FFFF}{c_2}}, \alpha_3, {\color{#FFFF55}{c_3}}}}_{({\color{#00FFFF}{\chi_2}} \times 16 \times {\color{#FFFF55}{\chi_3}})} \cdots \underbrace{G^{(N)}_{{\color{#FF9999}{c_{N-1}}}, \alpha_N, {\color{#BBFF66}{c_N}}}}_{({\color{#FF9999}{\chi_{N-1}}} \times 16 \times {\color{#BBFF66}{\chi_N}})} \cdot \underbrace{C^{(N)}_{{\color{#BBFF66}{c_N}}, {\color{#FF66FF}{y_N}}}}_{({\color{#BBFF66}{\chi_N}} \times {\color{#FF66FF}{1}})}  \end{aligned}$$

In [ ]:
class PathIntegralMap:
    """Computes L(t_k) = matrix of eps_{t_k,0} on the grid t_k = k*dt_fs."""

    def __init__(self, H_cm, dt_fs, kmax, lam_cm, gamma_cm, T_K, *, fs_to_cm, kb_cm, n_mats=200000, eps=1e-8, chi_max=256):
        # Allgemeine Systemparameter
        self.d = H_cm.shape[0]          # d = dim des Hamiltonin
        self.D = self.d ** 2            # dim des Liouville-Raums (d*d) für Superoperatoren
        self.kmax = int(kmax)           # Die maximale Gedächtnistiefe. System "merkt" sich nur Einflüsse der letzten kmax Zeitschritte
        self.eps = eps                  # Die Fehlerschwelle für das Abschneiden der SVD
        self.chi_max = chi_max          # Die maximale Bond-Dimension des Tensornetzwerks (=Zahl der größten Singulärwerte, die wir behalten)

        # Für das Influence Functional
        dtau = dt_fs * fs_to_cm         # Rechnet den Zeitschritt von Femtosekunden (dt_fs) in inverse Wellenzahlen
        beta = 1.0 / (kb_cm * T_K)      # Berechnet die thermodynamische inverse Temperatur
        a, nu = drude_expansion(lam_cm, gamma_cm, beta, n_mats)     # amplituden und zerfallsraten der Bad-Moden (Matsubara-Expansion)
        self.eta = eta_coefficients(a, nu, dtau, self.kmax)         # beschreiben, wie stark ein Zeitschritt t_k mit einem früheren Zeitschritt t_j über das Bad korreliert ist
        self.Imats = influence_matrices(self.eta, self.d)           # Gewichte, die in die MPS-Tensoren einmultipliziert werden
        self.I0 = np.diag(self.Imats[0]).copy()                     # Wechselwirkung eines Zeitschritts mit sich selbst. Da diese Matrix diagonal ist, extrahiert np.diag diese Diagonale als einfachen Vektor

        # Die Propagatoren für das System
        Uh = expm(-1j * H_cm * dtau / 2)          # Zeitentwicklungsoperator für einen halben Zeitschritt (Delta tau / 2)
        self.Ph = np.kron(Uh.conj(), Uh)          # half-step FB propagator: rho(t) = U rho U^t$ -> vec(rho(t)) = (U^* \otimes U) vec(rho(0))
        self.P = self.Ph @ self.Ph                # full-step FB propagator

    # -- read-out: contract MPS, apply trailing half propagator ------------
    def _readout(self, mps):
        """
        Zieht das gesamte Netzwerk zusammen (kontrahiert alle internen Bonds c),
        sodass nur noch alpha_0 (Anfang) und alpha_last (Ende) uebrig bleiben.
        Am Ende wird der End-Halbschritt Ph multipliziert, was exakt der
        Liouville-Propagator-Matrix (16x16) fuer diesen Zeitschritt entspricht.
        """
        # Hier wird das kollabierte Netzwerk von links nach rechts Schritt für Schritt aufkumuliert. Zu Beginn ist dieser Speicher noch leer.
        env = None

        # Wir starten eine Schleife durch die Matrix Product State (MPS) Tensoren – allerdings ohne den allerletzten Tensor ([:-1]). 
        # Jeder dieser Tensoren repräsentiert einen vergangenen Zeitschritt im Gedächtnisfenster. 
        for site in mps[:-1]:

            # Bedeutung: Jeder Tensor site in der MPS-Kette hat drei Indizes: [linker_bond, physikalischer_index, rechter_bond]. 
            # Der physikalischer_index (Achse 1) steht für die Zustände des Bads zu diesem Zeitpunkt. Da uns die genauen, mikroskopischen 
            # Zustände der Vergangenheit für das Endergebnis nicht mehr interessieren (wir wollen die totale Spur über die Umgebung bilden), 
            # summieren wir über diesen mittleren Index auf (.sum(axis=1)). Aus dem 3D-Tensor wird eine normale 2D-Matrix v mit den Dimensionen [linker_bond, rechter_bond].
            v = site.sum(axis=1)                  # sum physical index

            # Bedeutung: Hier werden die kollabierten Matrizen v von links nach rechts miteinander multipliziert. Am Ende dieser Schleife ist 
            # die gesamte Kette bis auf den letzten Tensor zu einer einzigen Matrix env zusammengeschrumpft, die die Dimension [Anfangs-Bond, aktueller_Bond] besitzt.
            env = v if env is None else env @ v

        # Jetzt holen wir den allerletzten Tensor der Kette (mps[-1]), den wir vorhin in der Schleife ausgelassen haben. Dieser repräsentiert 
        # den aktuellsten Zeitschritt (die Gegenwart). Da die Pfadintegral-Kette hier endet, ist der rechte virtuelle Bond ungenutzt und hat 
        # die Dimension 1. Wir schneiden diesen Index mit [, :, 0] ab. Übrig bleibt eine Matrix G mit den Dimensionen [linker_bond, physikalischer_index_der_gegenwart]. 
        # Der physikalische Index wird hier nicht aufsummiert, da wir die Dynamik für diesen aktuellen Zustand wissen wollen!
        G = mps[-1][:, :, 0]                      # (chi_l, alpha_last)

        # Wir multiplizieren unsere angesammelte Kette env von links an diesen gegenwärtigen Tensor G. Das Ergebnis Lpre ist nun eine 2D-Matrix, 
        # in der alle internen Verschränkungs-Bonds verschwunden sind. Es existieren nur noch zwei Indizes: Der allererste Index der Kette 
        # (alpha_0, der den Zustand ganz am Anfang der Simulation beschreibt) und der aktuelle Index (alpha_last, der Zustand im Hier und Jetzt).
        Lpre = G if env is None else env @ G      # (alpha_0, alpha_last)

        # Ganz am Ende multiplizieren wir von links den in der __init__ vorbereiteten Halbschritt-Propagator self.Ph hinzu. Das ist 
        # physikalisch notwendig, weil das QUAPI-Pfadintegral symmetrisch aufgebaut ist
        return self.Ph @ Lpre.T                   # L[beta, alpha_0]

    # -- one propagation step ----------------------------------------------
    # Gauge invariant: on entry the MPS is right-canonical with the
    # orthogonality centre at site 0 (this is what the final SVD sweep of
    # the previous step leaves behind).  The influence factors
    # I_D(alpha_new, alpha_old) and the creation of the new slice are then
    # applied in a single left-to-right zip-up pass, so every truncation
    # happens at a proper mixed-canonical cut.  The value l of the new
    # slice index is carried along the pass and closed at the right end.
    def _step(self, mps):
        """
        Die Herzstueck-Funktion ("Zip-Up" Algorithmus).
        Fuegt den neuen Zeitschritt alpha_k hinzu und baut die neuen
        Tensoren auf. Variablen exakt an die formale SVD-Kette (W, M, G, C) angepasst.
        """
        D = self.D
        N = len(mps)                                    # Die aktuelle Länge unserer MPS-Kette (n_old).
        
        # W^(N): Gewicht für den kürzesten Pfad-Schritt am rechten Rand der Formel
        W_2 = self.Imats[1] * self.P * self.I0[:, None]   

        if N == 1:
            # Spezialfall: Der allererste Schritt.
            W_1 = mps[0][:, :, 0]                       
            M_tensor = W_1[:, :, None] * W_2.T[None, :, :]   
            c_0, alpha_1, y_1 = M_tensor.shape
            
            # SVD[M^(1)] -> G^(1) und C^(1)
            G_1, s, Vh = _svd_split(M_tensor.reshape(c_0 * alpha_1, y_1), self.eps, self.chi_max)  
            C_1 = (s[:, None] * Vh)
            mps = [G_1.reshape(c_0, alpha_1, -1), C_1.reshape(-1, y_1, 1)]
        else:
            # Zip-up von Links nach Rechts durch das Gedaechtnis-Fenster
            W_1_state = mps[0]                          # Zuvor 'G', jetzt Teil des alten Zustands W^(1)
            W_1_env = self.Imats[N]                     # Zuvor 'F', jetzt Bad-Einfluss für W^(1)
            
            # M^(1) = W_1_state * W_1_env
            M_tensor = W_1_state[:, :, None, :] * W_1_env.T[None, :, :, None]     
            c_0, alpha_1, y_1, y_1_old = M_tensor.shape                  
            
            M_1 = M_tensor.reshape(c_0 * alpha_1, y_1 * y_1_old)
            
            # SVD[M^(1)] -> G^(1) und C^(1)
            G_1, s, Vh = _svd_split(M_1, self.eps, self.chi_max, rank_hint=y_1_old)  
            
            new_mps = [G_1.reshape(c_0, alpha_1, -1)]             
            C = (s[:, None] * Vh).reshape(-1, y_1, y_1_old)     # Das Gepäck C^(1)

            # middle sites 
            for pos in range(1, N - 1):             
                W_k_env = self.Imats[N - pos]           # Zuvor 'F', Bad-Einfluss für Zeitschritt k
                W_k_state = mps[pos]                    # Zuvor 'G', System-Zustand für Zeitschritt k
                
                # M^(k) = C^(k-1) * W^(k)_state * W^(k)_env
                M_tensor = np.tensordot(C, W_k_state, axes=([2], [0])) 
                M_tensor *= W_k_env[None, :, :, None]                
                
                c_prev, y_k, alpha_k, y_k_old = M_tensor.shape                    
                
                M_k = M_tensor.transpose(0, 2, 1, 3).reshape(c_prev * alpha_k, y_k * y_k_old)       
                
                # SVD[M^(k)] -> G^(k) und C^(k)
                G_k, s, Vh = _svd_split(M_k, self.eps, self.chi_max, rank_hint=y_k_old)  
                
                new_mps.append(G_k.reshape(c_prev, alpha_k, -1))     
                C = (s[:, None] * Vh).reshape(-1, y_k, y_k_old) 

            # last old site 
            W_N_state = mps[-1][:, :, 0]                             
            
            # M^(N) = C^(N-1) * W^(N)_state * W^(N)
            M_tensor = np.tensordot(C, W_N_state, axes=([2], [0]))          
            M_tensor *= W_2[None, :, :]                               
            
            c_prev, y_N, alpha_N = M_tensor.shape                                
            
            M_N = M_tensor.transpose(0, 2, 1).reshape(c_prev * alpha_N, y_N)       
            
            # SVD[M^(N)] -> G^(N) und C^(N)
            G_N, s, Vh = _svd_split(M_N, self.eps, self.chi_max)     
            
            new_mps.append(G_N.reshape(c_prev, alpha_N, -1))              
            new_mps.append((s[:, None] * Vh).reshape(-1, y_N, 1))  
            mps = new_mps

        # restore right-canonical gauge (SVD Sweep rueckwaerts)
        for i in range(len(mps) - 1, 0, -1):   
            c_left, alpha_k, c_right = mps[i].shape   
            G_rev, s, Vh = _svd_split(mps[i].reshape(c_left, alpha_k * c_right), self.eps, self.chi_max)
            mps[i] = Vh.reshape(-1, alpha_k, c_right)
            mps[i - 1] = np.tensordot(mps[i - 1], G_rev * s[None, :], axes=([2], [0]))      

        # memory truncation: marginalize slices older than kmax
        if len(mps) > self.kmax:            
            v = mps[0].sum(axis=1)          
            mps[1] = np.tensordot(v, mps[1], axes=([1], [0]))   
            mps.pop(0)                      
            
        return mps

    # -- driver --------------------------------------------------------------
    def run(self, K, verbose=True, callback=None):
        """Returns [L(0), L(dt), ..., L(K*dt)].
        callback(k, maps) is invoked after every step (checkpointing).

        Hauptschleife, die das System iterativ vorwaerts in der Zeit propagiert.
        """
        D = self.D
        maps = [np.eye(D, dtype=complex)]

        # first slice: A_1[alpha_0, a_1] = I0[a_1] * Ph[a_1, alpha_0]
        # Start-Halbschritt wird in den allerersten Tensor absorbiert
        T = (self.I0[:, None] * self.Ph).T
        mps = [T[:, :, None].copy()]        # I0 * Ph
        maps.append(self._readout(mps))     # Ph * I0 * Ph

        for k in range(2, K + 1):     # baut map aus K Schritten
            mps = self._step(mps)
            maps.append(self._readout(mps))

            if verbose and (k % 10 == 0 or k == K):
                chi = max(t.shape[2] for t in mps)
                print(f"  step {k:4d}/{K}   max bond = {chi}", flush=True)

            if callback is not None:
                callback(k, maps)

        return maps